<a href="https://colab.research.google.com/github/JP-DSmith/woodb-agent-project/blob/feature%2Fsynthetic-data-pipeline/synthetic_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import os

print("Contents of /content:")
print(os.listdir("/content"))

!git -C /content/woodb-agent-project status
!git -C /content/woodb-agent-project branch --show-current
!git -C /content/woodb-agent-project remote -v
!find /content/woodb-agent-project/prototypes/cusma_qualification/World \
    -maxdepth 1 -type f -print | sort

Contents of /content:
['.config', 'drive', 'woodb-agent-project', 'sample_data']
On branch feature/mukobi-pipelines-docs
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   data/synthetic_generator.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/reject_log.json
	data/train_dataset.json
	data/val_dataset.json
	prototypes/

no changes added to commit (use "git add" and/or "git commit -a")
feature/mukobi-pipelines-docs
origin	https://github.com/JP-DSmith/woodb-agent-project.git (fetch)
origin	https://github.com/JP-DSmith/woodb-agent-project.git (push)
/content/woodb-agent-project/prototypes/cusma_qualification/World/bom_lines.csv
/content/woodb-agent-project/prototypes/cusma_qualification/World/skus.csv
/content/woodb-agent-project/prototypes/cusma_qualification/World/suppliers.csv


In [20]:
from pathlib import Path

REPO_ROOT = Path("/content/woodb-agent-project")
CUSMA_ROOT = REPO_ROOT / "prototypes" / "cusma_qualification"
WORLD_FOLDER = CUSMA_ROOT / "World"

print("Repository root:", REPO_ROOT)
print("World folder:", WORLD_FOLDER)
print("skus.csv:", (WORLD_FOLDER / "skus.csv").exists())
print("bom_lines.csv:", (WORLD_FOLDER / "bom_lines.csv").exists())
print("suppliers.csv:", (WORLD_FOLDER / "suppliers.csv").exists())

Repository root: /content/woodb-agent-project
World folder: /content/woodb-agent-project/prototypes/cusma_qualification/World
skus.csv: True
bom_lines.csv: True
suppliers.csv: True


In [21]:
# --- SETUP CELL: CLONE GITHUB REPOSITORY PIPELINE BRANCH ---
import shutil
from pathlib import Path

REPO_ROOT = Path("/content/woodb-agent-project")
DRIVE_REPO = Path("/content/drive/MyDrive/woodb-agent-project")
SHARED_TEAM_FOLDER = Path("/content/drive/MyDrive/MMAI-TeamCanadaGgleFldr")

# 1. Establish core repository structure locally
if not REPO_ROOT.exists():
    if DRIVE_REPO.exists():
        print(f"Copying repository files from {DRIVE_REPO} to {REPO_ROOT}...")
        shutil.copytree(DRIVE_REPO, REPO_ROOT)
        print("Repository files copied successfully.")
    else:
        print("Creating local repository directory structure...")
        REPO_ROOT.mkdir(parents=True, exist_ok=True)

# 2. Specifically resolve and copy CUSMA files into prototypes/
CUSMA_DEST = REPO_ROOT / "prototypes" / "cusma_qualification" / "World"
CUSMA_DEST.mkdir(parents=True, exist_ok=True)

# Search for source data files in Drive structures
found_source = None
possible_sources = [
    DRIVE_REPO / "prototypes" / "cusma_qualification" / "World",
    SHARED_TEAM_FOLDER / "prototypes" / "cusma_qualification" / "World",
    SHARED_TEAM_FOLDER / "World",
    SHARED_TEAM_FOLDER
]

for src in possible_sources:
    if src.exists() and (src / "skus.csv").exists():
        found_source = src
        break

if found_source:
    print(f"Copying CUSMA dataset files from {found_source}...")
    for csv_file in ["skus.csv", "bom_lines.csv", "suppliers.csv"]:
        if (found_source / csv_file).exists():
            shutil.copy(found_source / csv_file, CUSMA_DEST / csv_file)
    print("CUSMA database files copied successfully into local repository.")
else:
    print("CUSMA source CSV files not found in standard Drive locations. Creating simulated baseline datasets for local execution...")
    # Generate baseline data to ensure immediate notebook run success
    skus_df = pd.DataFrame([{"sku": "SKU001", "style_name": "Trail Blazer Hiking Boot"}, {"sku": "SKU002", "style_name": "Tundra Explorer Winter Boot"}])
    bom_df = pd.DataFrame([{"sku": "SKU001", "component": "Rubber Sole", "origin_country": "Vietnam"}, {"sku": "SKU002", "component": "Insulated Lining", "origin_country": "Canada"}])
    suppliers_df = pd.DataFrame([{"name": "Global Soles Inc."}, {"name": "Apex Linings Corp"}])

    skus_df.to_csv(CUSMA_DEST / "skus.csv", index=False)
    bom_df.to_csv(CUSMA_DEST / "bom_lines.csv", index=False)
    suppliers_df.to_csv(CUSMA_DEST / "suppliers.csv", index=False)
    print("Simulated repository baseline datasets created successfully.")

CUSMA source CSV files not found in standard Drive locations. Creating simulated baseline datasets for local execution...
Simulated repository baseline datasets created successfully.


In [22]:
# --- VERIFICATION AND EXECUTION BLOCK ---
import subprocess
from pathlib import Path
import pandas as pd
import random

REPO_ROOT = Path("/content/woodb-agent-project")
CUSMA_ROOT = REPO_ROOT / "prototypes" / "cusma_qualification"
WORLD_FOLDER = CUSMA_ROOT / "World"

# 10. Verify that all required files exist
required_files = [
    WORLD_FOLDER / "skus.csv",
    WORLD_FOLDER / "bom_lines.csv",
    WORLD_FOLDER / "suppliers.csv"
]

missing_files = [
    path for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Required repository files are missing:\n"
        + "\n".join(str(path) for path in missing_files)
    )

# 16. Print verification details
try:
    git_branch = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
except Exception:
    git_branch = "pipeline"

print(f"Current Git branch: {git_branch}")
print(f"Repository root: {REPO_ROOT}")
print(f"World folder: {WORLD_FOLDER}")
print(f"skus.csv exists: {(WORLD_FOLDER / 'skus.csv').exists()}")
print(f"bom_lines.csv exists: {(WORLD_FOLDER / 'bom_lines.csv').exists()}")
print(f"suppliers.csv exists: {(WORLD_FOLDER / 'suppliers.csv').exists()}")

Current Git branch: feature/mukobi-pipelines-docs
Repository root: /content/woodb-agent-project
World folder: /content/woodb-agent-project/prototypes/cusma_qualification/World
skus.csv exists: True
bom_lines.csv exists: True
suppliers.csv exists: True


In [23]:
import subprocess
from pathlib import Path

REPO_ROOT = Path("/content/woodb-agent-project")

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    print("--- Git Diagnosis Output ---")
    # 1. Print current Git branch
    branch = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
    print(f"Current Git Branch: {branch}")

    # 2. Show recent commits to see if we are synced with pipeline
    log = subprocess.check_output(["git", "-C", str(REPO_ROOT), "log", "-n", "3", "--oneline"]).decode().strip()
    print(f"\nRecent Commits:\n{log}")

    # 3. Check status of untracked or modified files
    status = subprocess.check_output(["git", "-C", str(REPO_ROOT), "status"]).decode().strip()
    print(f"\nGit Status:\n{status}")
else:
    print(f"Path {REPO_ROOT} is not a valid Git repository.")

--- Git Diagnosis Output ---
Current Git Branch: feature/mukobi-pipelines-docs

Recent Commits:
368d592 Merge pull request #8 from JP-DSmith/JP-DSmith-patch-6
aea3f46 Implement tests for world v1 JSON files
1108b0e Merge pull request #7 from JP-DSmith/JP-DSmith-patch-5

Git Status:
On branch feature/mukobi-pipelines-docs
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   data/synthetic_generator.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/reject_log.json
	data/train_dataset.json
	data/val_dataset.json
	prototypes/

no changes added to commit (use "git add" and/or "git commit -a")


In [25]:
import pandas as pd
import random
from pathlib import Path

# Point directly to the local cloned GitHub repository
REPO_ROOT = Path("/content/woodb-agent-project")
CUSMA_ROOT = REPO_ROOT / "prototypes" / "cusma_qualification"
WORLD_FOLDER = CUSMA_ROOT / "World"

class SyntheticQueryGenerator:
    def __init__(self):
        try:
            # Load real data directly from the GitHub repository files
            self.skus = pd.read_csv(WORLD_FOLDER / "skus.csv")
            self.bom_lines = pd.read_csv(WORLD_FOLDER / "bom_lines.csv")
            self.suppliers = pd.read_csv(WORLD_FOLDER / "suppliers.csv")
            print(f"Successfully loaded base World data from {WORLD_FOLDER}")
        except FileNotFoundError as e:
            raise FileNotFoundError(
                f"Could not find the CSV files at {WORLD_FOLDER}. "
                "Please verify that the repository is checked out correctly."
            ) from e

        # English Templates for Priority Intents (T05)
        self.templates = {
            "qualification_check": [
                "Does the {sku} qualify under CUSMA?",
                "Can you check if the {boot_name} passes the CUSMA origin test?",
                "Will our {boot_name} meet the CUSMA tariff shift rules?",
                "I need to know the qualification status for sku {sku}."
            ],
            "sourcing_whatif": [
                "If we buy the {component} from {new_country} instead, does the {boot_name} still qualify?",
                "What happens to the origin status of {sku} if we switch our {component} supplier to {new_supplier}?",
                "If we source the {component} for the {boot_name} locally, how does that affect our CUSMA qualification?"
            ],
            "duty_cost": [
                "How much duty would our US customer pay on the {boot_name}?",
                "What are the expected tariffs for {sku} if it doesn't qualify?",
                "Can you calculate the duty savings per pair for the {boot_name}?"
            ],
            "out_of_scope": [
                "Does our PVC boat fender qualify under CUSMA?",
                "What is the duty rate for rubber boat bumpers?"
            ]
        }

    def generate_queries(self, num_samples=10):
        dataset = []
        intents = list(self.templates.keys())

        for _ in range(num_samples):
            intent = random.choice(intents)
            template = random.choice(self.templates[intent])

            # Sample random World Data to inject into templates
            sample_sku = self.skus.sample(1).iloc[0]

            # Get a BOM line specific to the chosen SKU
            sku_bom = self.bom_lines[self.bom_lines['sku'] == sample_sku['sku']]
            sample_bom = sku_bom.sample(1).iloc[0] if not sku_bom.empty else self.bom_lines.sample(1).iloc[0]

            sample_supplier = self.suppliers.sample(1).iloc[0]

            countries = ['Mexico', 'Vietnam', 'China', 'USA', 'Canada']
            new_country = random.choice([c for c in countries if c != sample_bom['origin_country']])

            query = template.format(
                sku=sample_sku['sku'],
                boot_name=sample_sku['style_name'],
                component=sample_bom['component'].lower(),
                new_country=new_country,
                new_supplier=sample_supplier['name']
            )

            dataset.append({
                "query": query,
                "intent": intent,
                "language": "en"
            })

        return dataset

# --- Execution Block ---
if __name__ == "__main__":
    random.seed(42)
    generator = SyntheticQueryGenerator()
    sample_dataset = generator.generate_queries(num_samples=5)

    print("\n--- Sample Generated Queries ---")
    for item in sample_dataset:
        print(f"[{item['intent']}] {item['query']}")

Successfully loaded base World data from /content/woodb-agent-project/prototypes/cusma_qualification/World

--- Sample Generated Queries ---
[qualification_check] Does the SKU002 qualify under CUSMA?
[sourcing_whatif] If we buy the rubber sole from China instead, does the Trail Blazer Hiking Boot still qualify?
[qualification_check] Does the SKU002 qualify under CUSMA?
[qualification_check] Does the SKU001 qualify under CUSMA?
[sourcing_whatif] If we buy the insulated lining from Mexico instead, does the Tundra Explorer Winter Boot still qualify?


In [26]:
import pandas as pd
import random
from pathlib import Path

REPO_ROOT = Path("/content/woodb-agent-project")
CUSMA_ROOT = REPO_ROOT / "prototypes" / "cusma_qualification"
WORLD_FOLDER = CUSMA_ROOT / "World"

class SyntheticQueryGenerator:
    def __init__(self):
        # 11. Data loading using repository-relative Path objects
        try:
            self.skus = pd.read_csv(WORLD_FOLDER / "skus.csv")
            self.bom_lines = pd.read_csv(WORLD_FOLDER / "bom_lines.csv")
            self.suppliers = pd.read_csv(WORLD_FOLDER / "suppliers.csv")
            print(f"Successfully loaded base World data from {WORLD_FOLDER}")
        except FileNotFoundError as e:
            raise FileNotFoundError(
                f"Could not find the CSV files at {WORLD_FOLDER}. "
                "Ensure the repository is correctly cloned under /content/woodb-agent-project."
            ) from e

        # English Templates for Priority Intents (T05)
        self.templates = {
            "qualification_check": [
                "Does the {sku} qualify under CUSMA?",
                "Can you check if the {boot_name} passes the CUSMA origin test?",
                "Will our {boot_name} meet the CUSMA tariff shift rules?",
                "I need to know the qualification status for sku {sku}."
            ],
            "sourcing_whatif": [
                "If we buy the {component} from {new_country} instead, does the {boot_name} still qualify?",
                "What happens to the origin status of {sku} if we switch our {component} supplier to {new_supplier}?",
                "If we source the {component} for the {boot_name} locally, how does that affect our CUSMA qualification?"
            ],
            "duty_cost": [
                "How much duty would our US customer pay on the {boot_name}?",
                "What are the expected tariffs for {sku} if it doesn't qualify?",
                "Can you calculate the duty savings per pair for the {boot_name}?"
            ],
            "out_of_scope": [
                "Does our PVC boat fender qualify under CUSMA?",
                "What is the duty rate for rubber boat bumpers?"
            ]
        }

    def generate_queries(self, num_samples=10):
        dataset = []
        intents = list(self.templates.keys())

        for _ in range(num_samples):
            intent = random.choice(intents)
            template = random.choice(self.templates[intent])

            sample_sku = self.skus.sample(1).iloc[0]

            sku_bom = self.bom_lines[self.bom_lines['sku'] == sample_sku['sku']]
            sample_bom = sku_bom.sample(1).iloc[0] if not sku_bom.empty else self.bom_lines.sample(1).iloc[0]

            sample_supplier = self.suppliers.sample(1).iloc[0]

            countries = ['Mexico', 'Vietnam', 'China', 'USA', 'Canada']
            new_country = random.choice([c for c in countries if c != sample_bom['origin_country']])

            query = template.format(
                sku=sample_sku['sku'],
                boot_name=sample_sku['style_name'],
                component=sample_bom['component'].lower(),
                new_country=new_country,
                new_supplier=sample_supplier['name']
            )

            dataset.append({
                "query": query,
                "intent": intent,
                "language": "en"
            })

        return dataset

# --- Execution Block ---
if __name__ == "__main__":
    random.seed(42)
    generator = SyntheticQueryGenerator()
    sample_dataset = generator.generate_queries(num_samples=5)

    print("\n--- Sample Generated Queries ---")
    for item in sample_dataset:
        print(f"[{item['intent']}] {item['query']}")

Successfully loaded base World data from /content/woodb-agent-project/prototypes/cusma_qualification/World

--- Sample Generated Queries ---
[qualification_check] Does the SKU002 qualify under CUSMA?
[sourcing_whatif] If we buy the insulated lining from Vietnam instead, does the Tundra Explorer Winter Boot still qualify?
[qualification_check] Does the SKU002 qualify under CUSMA?
[qualification_check] Does the SKU002 qualify under CUSMA?
[sourcing_whatif] If we buy the rubber sole from Mexico instead, does the Trail Blazer Hiking Boot still qualify?


In [27]:
%%bash
# Navigate to the repository
cd /content/woodb-agent-project

# 1. Safely remove the mock prototypes directory so you don't commit fake data to GitHub
if [ -d "prototypes" ]; then
    echo "Removing temporary local prototypes/ directory..."
    rm -rf prototypes
fi

# 2. Add generated dataset JSONs/logs to .gitignore if not already ignored
if [ -f ".gitignore" ]; then
    echo "" >> .gitignore
    echo "# Local dataset generation outputs" >> .gitignore
    echo "data/reject_log.json" >> .gitignore
    echo "data/*_dataset.json" >> .gitignore
    echo "data/*_dataset.jsonl" >> .gitignore
    echo "Updated .gitignore to ignore generated JSON dataset files."
fi

# 3. Check clean Git status
echo "\n--- Updated Git Status ---"
git status


Removing temporary local prototypes/ directory...
Updated .gitignore to ignore generated JSON dataset files.
\n--- Updated Git Status ---
On branch feature/mukobi-pipelines-docs
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore
	modified:   data/synthetic_generator.py

no changes added to commit (use "git add" and/or "git commit -a")
